# Competitive Research Copilot — P4 Colab Runner

**Run the cells from top to bottom.** This notebook locates the extracted project folder automatically, so it will not assume a specific ZIP folder name.

Current scope: P4 tests and graph/tool smoke test. Live P3 retrieval and full agent run remain pending their integration sources/configuration.

## 1 — Upload the P4 ZIP
Upload `Competitive_Research_Copilot_P4_Complete.zip` when prompted.

In [ ]:
from google.colab import files
import os, zipfile, pathlib, shutil
uploaded = files.upload()
if not uploaded:
    raise RuntimeError('No ZIP uploaded. Please upload the P4 project ZIP and rerun this cell.')
zip_path = next(iter(uploaded))
if not zip_path.lower().endswith('.zip'):
    raise ValueError('Please upload the project ZIP file, not the notebook or another file.')
extract_dir = '/content/p4_extracted'
shutil.rmtree(extract_dir, ignore_errors=True)
os.makedirs(extract_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as z:
    z.extractall(extract_dir)
# Locate project root by its distinctive source files, regardless of wrapper folder name.
roots = [p for p in pathlib.Path(extract_dir).rglob('graph.py') if p.parent.name == 'p4' and (p.parent.parent / 'requirements.txt').exists()]
if not roots:
    raise FileNotFoundError('Could not locate P4 project root (expected p4/graph.py and requirements.txt inside ZIP).')
project_dir = roots[0].parent.parent
os.chdir(project_dir)
print('Project detected:', project_dir)
print('Project files:', sorted(os.listdir(project_dir))[:30])

## 2 — Install dependencies

In [ ]:
%pip install -q -r requirements.txt

## 3 — Run automated tests

In [ ]:
!PYTHONPATH=. pytest -v

## 4 — Check graph and tool registry
This is a graph-construction smoke test using dummy callbacks; it does not simulate a live model research run.

In [ ]:
from p4.graph import build_graph
from p4.tools.registry import build_tool_registry
from p4.tools.retrieval import RetrievalAdapter
adapter = RetrievalAdapter()
tools = build_tool_registry(adapter)
dummy = lambda *args, **kwargs: {}
graph = build_graph(tools, planner_fn=dummy, researcher_fn=dummy, analyst_fn=dummy, writer_fn=dummy, critic_fn=dummy)
print('Graph construction: OK')
print('Graph type:', type(graph).__name__)
print('Registered tool count:', len(tools))
print('Registered tools:', list(tools.keys()))

## 5 — Retrieval adapter status
The P3 adapter is intentionally unconfigured until the actual P3 retrieval source is connected.

In [ ]:
print(adapter.search_reports('HUL advertising and promotion expenses FY24', k=3, filters={'company':'HUL','fiscal_year':'FY24'}))

## 6 — Connect the supplied P3 retrieval source and index
The P3 source is included in this package. Upload the P3 index ZIP and the required parsed corpus file (`all_chunks.jsonl`) when prompted. The index ZIP by itself is insufficient because P3 source also loads parsed chunks.


In [ ]:
from google.colab import files
import zipfile, os, pathlib, shutil
p3_dir = pathlib.Path(project_dir) / "p3_retrieval"
index_upload = files.upload()  # Upload the P3 index ZIP
if index_upload:
    index_zip = next(iter(index_upload))
    with zipfile.ZipFile(index_zip) as z:
        z.extractall(p3_dir)
    print("Index extracted into", p3_dir)
else:
    print("No index ZIP uploaded; skipping live retrieval setup.")


### Upload parsed chunks
Upload the P3 parsed corpus file named `all_chunks.jsonl` (typically from P3 `parsed/`). This is required by the supplied `retrieve.py`.


In [ ]:
parsed_upload = files.upload()  # Upload all_chunks.jsonl
if parsed_upload:
    parsed_name = next(iter(parsed_upload))
    if not parsed_name.endswith("all_chunks.jsonl"):
        print("Warning: expected all_chunks.jsonl; uploaded:", parsed_name)
    parsed_dir = p3_dir / "parsed"
    parsed_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy(parsed_name, parsed_dir / "all_chunks.jsonl")
else:
    print("Parsed corpus not uploaded; live P3 retrieval remains unavailable.")


### Install P3 retrieval dependencies
This may download large model/runtime packages.


In [ ]:
%pip install -q -r p3_retrieval/requirements.txt


### Configure and smoke-test P3 retrieval
This test imports the real P3 source and executes a query. Model downloads and/or API configuration may be required.


In [ ]:
from p4.tools.retrieval import RetrievalAdapter
p3_ready = (p3_dir / "index" / "bm25.pkl").exists() and (p3_dir / "index" / "faiss.index").exists() and (p3_dir / "index" / "chunk_ids.json").exists() and (p3_dir / "parsed" / "all_chunks.jsonl").exists()
if p3_ready:
    adapter = RetrievalAdapter.from_p3_path(str(p3_dir))
    live = adapter.search_reports("HUL advertising and promotion expenses FY24", k=3, filters={"company":"HUL", "fiscal_year":"FY24"})
    print(live["status"], "records:", len(live.get("records", [])))
    if live.get("records"):
        print(live["records"][0].get("doc_id"), live["records"][0].get("page"))
else:
    print("P3 retrieval not ready: verify index files and parsed/all_chunks.jsonl are uploaded.")


## Updated integration boundary
- P3 `retrieve.py` is now included in this P4 package.
- P3 index ZIP is provided separately and can be uploaded into the notebook.
- **Required but not yet supplied:** `parsed/all_chunks.jsonl`.
- Live retrieval is only confirmed if the final smoke-test cell returns records.
- P5 observability/deployment remains pending.


## Current integration limitations
- P4 tests and graph smoke test are available.
- P3 retrieval source is included; the supplied index ZIP can be added in the P3 setup cells.
- Live P3 retrieval still requires `parsed/all_chunks.jsonl` and a successful smoke test.
- P5 tracing, evaluation runner, app and deployment remain pending.
- Do not claim a full end-to-end research run until the live query and agent workflow are verified.
